
# 03 — Prepare the combined clinical-chemistry analysis matrix

Chemistry requires an explicit mapping and unit-review gate. The notebook reuses your existing curated VAE strata, standardizes them to generic feature names, and identifies any new linear features/components that still need review.

It **does not silently auto-select measurement concepts or mix units**. Missing mappings are written to:

- `working/chemistry_mapping_candidates_for_review.tsv`
- `working/chemistry_component_availability.tsv`

Add approved rows to `working/chemistry_strata_additions.tsv` with:

- `final_action = include_for_normalized_combining`
- a reviewed target unit and conversion factor.

Re-run this notebook after editing that file. Features with complete curated components proceed immediately; incomplete features remain in the availability audit and are not modeled.


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:

import re
import numpy as np
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

targets_file = cfg.TARGET_DIR / "analysis_targets_grouped.tsv"
shared_flags_file = cfg.SHARED_DIR / "aou_disease_flag_hits.tsv.gz"
pcs_file = cfg.SHARED_DIR / "aou_genetic_ancestry_pcs.tsv"
additions_file = cfg.WORKING_DIR / "chemistry_strata_additions.tsv"

for required in [targets_file, shared_flags_file, pcs_file, additions_file]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing {required}. Run earlier notebooks or restore the template."
        )

chemistry_spec = aru.load_chemistry_feature_spec(
    cfg.CONFIG_DIR / "chemistry_feature_spec.tsv"
)
targets = pd.read_csv(targets_file, sep="	")
chemistry_targets = targets[
    targets["omic_type"].eq("chemistries")
    & targets["clinically_testable"].fillna(False)
].copy()
target_features = sorted(
    chemistry_targets["chemistry_canonical_feature"].dropna().unique()
)

print(f"Testable grouped chemistry targets: {len(chemistry_targets):,}")
print(f"Unique canonical chemistry features: {len(target_features):,}")
print(target_features)


In [ ]:

# Start with the previously reviewed VAE strata, then append reviewed additions.
legacy_strata_file = cfg.first_existing(
    cfg.LEGACY_CHEMISTRY_STRATA_CANDIDATES,
    required=False,
)
selected_parts = []

if legacy_strata_file is not None:
    legacy = pd.read_csv(legacy_strata_file, sep="	")
    selected_parts.append(
        aru.normalize_selected_strata(legacy, chemistry_spec)
    )
    print("Loaded legacy curated strata:", legacy_strata_file)
else:
    print("WARNING: no legacy chemistry strata file was found.")

additions = pd.read_csv(additions_file, sep="	", keep_default_na=True)
included_additions = additions[
    additions["final_action"].astype(str).eq(
        "include_for_normalized_combining"
    )
].copy()
if not included_additions.empty:
    selected_parts.append(
        aru.normalize_selected_strata(included_additions, chemistry_spec)
    )

if selected_parts:
    selected_strata = pd.concat(selected_parts, ignore_index=True).drop_duplicates()
else:
    selected_strata = pd.DataFrame(
        columns=[
            "source_feature",
            "component",
            "measurement_concept_id",
            "unit",
            "target_unit",
            "conversion_factor_to_target",
            "conversion_offset_to_target",
        ]
    )

selected_strata.to_csv(
    cfg.WORKING_DIR / "chemistry_selected_strata_combined.tsv",
    sep="	",
    index=False,
)

component_availability = aru.chemistry_component_availability(
    target_features,
    chemistry_spec,
    selected_strata,
)
component_availability.to_csv(
    cfg.WORKING_DIR / "chemistry_component_availability.tsv",
    sep="	",
    index=False,
)
display(component_availability.sort_values("all_components_available"))


In [ ]:

# Build a searchable AoU measurement dictionary and a review table for gaps.
client = bigquery.Client(project=cfg.GOOGLE_PROJECT)
dictionary_out = cfg.CHEMISTRY_DIR / "aou_measurement_dictionary.tsv"

dictionary_query = f"""
SELECT
  m.measurement_concept_id,
  c.concept_name,
  c.concept_code,
  c.vocabulary_id,
  COALESCE(u.concept_name, 'Missing unit') AS unit,
  COUNT(*) AS n_records,
  COUNT(DISTINCT m.person_id) AS n_people
FROM `{cfg.CDR}.measurement` m
LEFT JOIN `{cfg.CDR}.concept` c
  ON m.measurement_concept_id = c.concept_id
LEFT JOIN `{cfg.CDR}.concept` u
  ON m.unit_concept_id = u.concept_id
GROUP BY 1,2,3,4,5
ORDER BY n_people DESC
"""
(cfg.LOG_DIR / "chemistry_measurement_dictionary.sql").write_text(
    dictionary_query, encoding="utf-8"
)

if dictionary_out.exists() and not cfg.FORCE_REBUILD:
    dictionary = pd.read_csv(dictionary_out, sep="	")
else:
    dictionary = client.query(dictionary_query).to_dataframe()
    dictionary.to_csv(dictionary_out, sep="	", index=False)

dictionary["concept_key"] = dictionary["concept_name"].fillna("").map(
    aru.normalize_text
)

missing_features = component_availability.loc[
    ~component_availability["all_components_available"],
    "canonical_feature",
].tolist()
spec_by_feature = chemistry_spec.set_index("canonical_feature")
review_rows = []

required_components = aru.required_chemistry_components(
    missing_features,
    chemistry_spec,
)
for req in required_components.itertuples(index=False):
    feature_row = spec_by_feature.loc[req.canonical_feature]
    search_terms = str(feature_row["search_terms"]).split("|")
    for term in search_terms:
        term_key = aru.normalize_text(term)
        if not term_key:
            continue
        matches = dictionary[
            dictionary["concept_key"].str.contains(
                re.escape(term_key), regex=True, na=False
            )
        ].head(75)
        for match in matches.itertuples(index=False):
            review_rows.append(
                {
                    "canonical_feature": req.canonical_feature,
                    "source_feature": req.source_feature,
                    "component": req.component,
                    "search_term": term,
                    "measurement_concept_id": match.measurement_concept_id,
                    "concept_name": match.concept_name,
                    "concept_code": match.concept_code,
                    "vocabulary_id": match.vocabulary_id,
                    "unit": match.unit,
                    "n_people": match.n_people,
                    "n_records": match.n_records,
                    "recommended_target_unit": feature_row["target_unit"],
                    "include_for_review": False,
                    "conversion_factor_to_target": np.nan,
                    "conversion_offset_to_target": 0.0,
                    "review_notes": "",
                }
            )

review = pd.DataFrame(review_rows).drop_duplicates(
    subset=[
        "source_feature",
        "component",
        "measurement_concept_id",
        "unit",
    ]
) if review_rows else pd.DataFrame()
review_out = cfg.WORKING_DIR / "chemistry_mapping_candidates_for_review.tsv"
review.to_csv(review_out, sep="	", index=False)
print("Saved mapping-review candidates:", review_out)
print("Missing canonical features:", missing_features)


In [ ]:

# Continue with every feature whose complete component set is currently curated.
available_features = component_availability.loc[
    component_availability["all_components_available"],
    "canonical_feature",
].tolist()

if not available_features:
    raise RuntimeError(
        "No chemistry target has a complete curated component set. Review "
        "working/chemistry_mapping_candidates_for_review.tsv and update "
        "working/chemistry_strata_additions.tsv."
    )

required_available = aru.required_chemistry_components(
    available_features,
    chemistry_spec,
)[["source_feature", "component"]].drop_duplicates()
selected_for_query = selected_strata.merge(
    required_available,
    on=["source_feature", "component"],
    how="inner",
    validate="many_to_one",
).drop_duplicates()

selected_cte = aru.build_selected_strata_cte(selected_for_query)
print(f"Available chemistry features: {len(available_features)}")
print(f"Curated concept-unit strata used: {len(selected_for_query)}")


In [ ]:

component_out = cfg.CHEMISTRY_DIR / "chemistry_person_component_long.tsv.gz"

component_query = f"""
WITH
{selected_cte},
raw_rows AS (
  SELECT
    m.person_id,
    s.source_feature,
    s.component,
    s.measurement_concept_id,
    s.unit AS selected_unit,
    s.target_unit,
    m.measurement_date,
    (
      m.value_as_number * s.conversion_factor_to_target
      + s.conversion_offset_to_target
    ) AS value_converted,
    CONCAT(
      s.source_feature, '||', s.component, '||',
      CAST(s.measurement_concept_id AS STRING), '||', s.unit
    ) AS stratum_id
  FROM `{cfg.CDR}.measurement` m
  LEFT JOIN `{cfg.CDR}.concept` u
    ON m.unit_concept_id = u.concept_id
  INNER JOIN selected_strata s
    ON m.measurement_concept_id = s.measurement_concept_id
   AND COALESCE(u.concept_name, 'Missing unit') = s.unit
  WHERE m.value_as_number IS NOT NULL
),
quantiles AS (
  SELECT
    stratum_id,
    APPROX_QUANTILES(value_converted, 100)[OFFSET(1)] AS q01,
    APPROX_QUANTILES(value_converted, 100)[OFFSET(99)] AS q99
  FROM raw_rows
  GROUP BY stratum_id
),
winsorized AS (
  SELECT
    r.*,
    CASE
      WHEN q.q01 IS NULL OR q.q99 IS NULL OR q.q01 >= q.q99
        THEN r.value_converted
      WHEN r.value_converted < q.q01 THEN q.q01
      WHEN r.value_converted > q.q99 THEN q.q99
      ELSE r.value_converted
    END AS value_winsorized
  FROM raw_rows r
  LEFT JOIN quantiles q USING (stratum_id)
),
stratum_stats AS (
  SELECT
    stratum_id,
    AVG(value_winsorized) AS stratum_mean,
    STDDEV(value_winsorized) AS stratum_sd
  FROM winsorized
  GROUP BY stratum_id
),
z_rows AS (
  SELECT
    w.*,
    CASE
      WHEN s.stratum_sd IS NULL OR s.stratum_sd = 0 THEN NULL
      ELSE (w.value_winsorized - s.stratum_mean) / s.stratum_sd
    END AS value_z
  FROM winsorized w
  LEFT JOIN stratum_stats s USING (stratum_id)
)
SELECT
  person_id,
  source_feature,
  component,
  ANY_VALUE(target_unit) AS target_unit,
  COUNT(*) AS n_measurements,
  MIN(measurement_date) AS first_measurement_date,
  MAX(measurement_date) AS last_measurement_date,
  APPROX_QUANTILES(value_winsorized, 100)[OFFSET(50)] AS value_raw_median,
  APPROX_QUANTILES(value_z, 100)[OFFSET(50)] AS value_z_median
FROM z_rows
WHERE value_z IS NOT NULL
GROUP BY person_id, source_feature, component
"""
(cfg.LOG_DIR / "chemistry_person_component_long.sql").write_text(
    component_query, encoding="utf-8"
)

if component_out.exists() and not cfg.FORCE_REBUILD:
    component_long = pd.read_csv(component_out, sep="	", compression="gzip")
else:
    component_long = client.query(component_query).to_dataframe()
    component_long.to_csv(
        component_out,
        sep="	",
        index=False,
        compression="gzip",
    )

component_long["person_id"] = pd.to_numeric(
    component_long["person_id"], errors="raise"
).astype("int64")
print("Person-component table:", component_long.shape)


In [ ]:

feature_matrix, feature_metadata = aru.assemble_chemistry_feature_matrix(
    component_long,
    available_features,
    chemistry_spec,
)
feature_metadata.to_csv(
    cfg.CHEMISTRY_DIR / "chemistry_feature_availability.tsv",
    sep="	",
    index=False,
)

usable_features = feature_metadata.loc[
    feature_metadata["status"].eq("ok"), "canonical_feature"
].tolist()
if not usable_features:
    raise RuntimeError("Curated chemistry strata produced no usable feature values")

print("Chemistry feature matrix:", feature_matrix.shape)
display(feature_metadata.sort_values(["status", "n_people"], ascending=[True, False]))


In [ ]:

# Query median age at the selected chemistry measurements and sex at birth.
covariates_out = cfg.CHEMISTRY_DIR / "chemistry_person_covariates.tsv"
covariates_query = f"""
WITH
{selected_cte},
selected_measurements AS (
  SELECT DISTINCT
    m.person_id,
    m.measurement_date,
    m.measurement_concept_id
  FROM `{cfg.CDR}.measurement` m
  LEFT JOIN `{cfg.CDR}.concept` u
    ON m.unit_concept_id = u.concept_id
  INNER JOIN selected_strata s
    ON m.measurement_concept_id = s.measurement_concept_id
   AND COALESCE(u.concept_name, 'Missing unit') = s.unit
  WHERE m.value_as_number IS NOT NULL
    AND m.measurement_date IS NOT NULL
),
people AS (
  SELECT
    p.person_id,
    p.year_of_birth,
    p.birth_datetime,
    p.sex_at_birth_concept_id,
    sex.concept_name AS sex_at_birth,
    CASE
      WHEN p.birth_datetime IS NOT NULL THEN DATE(p.birth_datetime)
      WHEN p.year_of_birth IS NOT NULL THEN DATE(p.year_of_birth, 7, 1)
      ELSE NULL
    END AS birth_date
  FROM `{cfg.CDR}.person` p
  LEFT JOIN `{cfg.CDR}.concept` sex
    ON p.sex_at_birth_concept_id = sex.concept_id
),
ages AS (
  SELECT
    sm.person_id,
    p.year_of_birth,
    p.birth_datetime,
    p.sex_at_birth_concept_id,
    p.sex_at_birth,
    sm.measurement_date,
    DATE_DIFF(sm.measurement_date, p.birth_date, DAY) / 365.25 AS age_years
  FROM selected_measurements sm
  JOIN people p USING (person_id)
  WHERE p.birth_date IS NOT NULL
)
SELECT
  person_id,
  ANY_VALUE(year_of_birth) AS year_of_birth,
  ANY_VALUE(birth_datetime) AS birth_datetime,
  ANY_VALUE(sex_at_birth_concept_id) AS sex_at_birth_concept_id,
  ANY_VALUE(sex_at_birth) AS sex_at_birth,
  APPROX_QUANTILES(age_years, 100)[OFFSET(50)] AS age_at_measurement,
  MIN(measurement_date) AS first_selected_measurement_date,
  MAX(measurement_date) AS last_selected_measurement_date,
  COUNT(*) AS n_selected_measurements
FROM ages
GROUP BY person_id
"""
(cfg.LOG_DIR / "chemistry_person_covariates.sql").write_text(
    covariates_query, encoding="utf-8"
)

if covariates_out.exists() and not cfg.FORCE_REBUILD:
    covariates = pd.read_csv(covariates_out, sep="	")
else:
    covariates = client.query(covariates_query).to_dataframe()
    covariates.to_csv(covariates_out, sep="	", index=False)

covariates["person_id"] = pd.to_numeric(
    covariates["person_id"], errors="raise"
).astype("int64")
covariates["age_at_measurement"] = pd.to_numeric(
    covariates["age_at_measurement"], errors="coerce"
)
covariates.loc[
    ~covariates["age_at_measurement"].between(18, 110),
    "age_at_measurement",
] = np.nan

sex_lower = covariates["sex_at_birth"].fillna("").astype(str).str.lower()
covariates["sex_adjustment"] = np.select(
    [sex_lower.eq("female"), sex_lower.eq("male")],
    ["Female", "Male"],
    default="Other/Unknown",
)


In [ ]:

flag_hits = pd.read_csv(shared_flags_file, sep="	", compression="gzip")
flag_hits["person_id"] = pd.to_numeric(
    flag_hits["person_id"], errors="raise"
).astype("int64")
disease_defs = aru.load_disease_definitions(cfg.CONFIG_DIR / "disease_definitions.json")
base_domains = [item["domain"] for item in disease_defs]
flags = aru.add_case_control_flags(covariates, flag_hits, base_domains)

pcs = pd.read_csv(pcs_file, sep="	")
pcs["person_id"] = pd.to_numeric(pcs["person_id"], errors="raise").astype("int64")

matrix = (
    feature_matrix.merge(covariates, on="person_id", how="left", validate="one_to_one")
    .merge(flags, on="person_id", how="left", validate="one_to_one")
    .merge(pcs, on="person_id", how="left", validate="one_to_one")
)
matrix["has_complete_pc8"] = matrix[cfg.PC_COLUMNS].notna().all(axis=1)

matrix_out = cfg.CHEMISTRY_DIR / "chemistry_analysis_matrix.tsv.gz"
matrix.to_csv(matrix_out, sep="	", index=False, compression="gzip")

# Map every chemistry target key to its canonical outcome column, even when
# unavailable; the model notebook then records feature_not_available explicitly.
all_feature_rows = chemistry_spec[
    chemistry_spec["canonical_feature"].isin(target_features)
][["canonical_feature", "display_label"]].copy()
all_feature_rows["feature_key"] = "chem:" + all_feature_rows["canonical_feature"]
all_feature_rows["outcome_col"] = (
    "chem_" + all_feature_rows["canonical_feature"] + "_z"
)
available_columns = set(matrix.columns)
all_feature_rows["available_for_model"] = all_feature_rows["outcome_col"].isin(
    available_columns
)
all_feature_rows.to_csv(
    cfg.CHEMISTRY_DIR / "chemistry_feature_column_map.tsv",
    sep="	",
    index=False,
)

print("Saved matrix:", matrix_out)
print("Matrix shape:", matrix.shape)
print("Complete PC8:", int(matrix["has_complete_pc8"].sum()))
print("Features available for modeling:", int(all_feature_rows["available_for_model"].sum()))
display(all_feature_rows.sort_values("available_for_model"))


In [ ]:

# Cohort-specific case/control counts for the primary PC-complete sample.
summary_rows = []
pc_mask = matrix["has_complete_pc8"]
for domain in aru.CASE_CONTROL_DOMAINS:
    summary_rows.append(
        {
            "analysis_domain": domain,
            "n_total": len(matrix),
            "n_cases": int(matrix[f"{domain}_case"].sum()),
            "n_domain_controls": int(
                matrix[f"{domain}_domain_control_candidate"].sum()
            ),
            "n_cases_complete_pc8": int(
                matrix.loc[pc_mask, f"{domain}_case"].sum()
            ),
            "n_domain_controls_complete_pc8": int(
                matrix.loc[pc_mask, f"{domain}_domain_control_candidate"].sum()
            ),
        }
    )
cohort_summary = pd.DataFrame(summary_rows)
cohort_summary.to_csv(
    cfg.CHEMISTRY_DIR / "chemistry_case_control_summary.tsv",
    sep="	",
    index=False,
)
display(cohort_summary)
